# Brain Tumor Gene Expression Classifier

## Data Loading
Loading TCGA gene expression and clinical phenotype files for 
both GBM and LGG cohorts.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

initial_df = pd.read_csv("data/TCGA-GBM.star_fpkm.tsv", sep="\t")
phenotype_df = pd.read_csv("data/TCGA-GBM.clinical.tsv", sep="\t")
lgg_df = pd.read_csv("data/TCGA-LGG.star_fpkm.tsv", sep="\t")
lgg_Phenotype_df = pd.read_csv("data/TCGA-LGG.clinical.tsv", sep="\t")


## Combining and Cleaning Data
Concatenating GBM and LGG expression matrices on shared genes, 
transposing so samples are rows, and merging with phenotype labels. 
Dropping rows with missing diagnosis values.

In [ ]:
lggGBM_Phenotype_df = pd.concat([phenotype_df[["sample", "primary_diagnosis.diagnoses"]], lgg_Phenotype_df[["sample", "primary_diagnosis.diagnoses"]]])
initial_df["Ensembl_ID"].is_unique
genomic_df = pd.concat([initial_df.set_index("Ensembl_ID"), lgg_df.set_index("Ensembl_ID")], axis=1)
genomic_T = genomic_df.T
genomic_T = genomic_T.reset_index().rename(columns={"index": "sample"})
merged_df = genomic_T.merge(lggGBM_Phenotype_df, on="sample")
merged_df = merged_df.dropna(subset=['primary_diagnosis.diagnoses'])

## Building Features and Labels
Separating into features (X) and integer-encoded labels (y) using 
LabelEncoder. PyTorch's cross entropy loss requires integer class 
labels rather than strings.

In [ ]:


le = LabelEncoder()
X = merged_df.drop(columns=["sample", "primary_diagnosis.diagnoses"])
y = le.fit_transform(merged_df["primary_diagnosis.diagnoses"])
test_set_size = int(len(y) * 0.3)


## Train/Test Split and Feature Selection
Manually splitting into 70/30 train/test using shuffled indices. 
Reducing 60,660 genes to the top 2,000 most informative features 
using ANOVA F-test. Selection is fit on training data only to avoid 
data leakage into the test set.

In [ ]:
import torch
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif


rp = torch.randperm(len(y)).tolist()
train_y = [y[i] for i in rp[:-test_set_size]]
test_y = [y[i] for i in rp[-test_set_size:]]
train_X = X.iloc[rp[:-test_set_size]]
test_X = X.iloc[rp[-test_set_size:]]
selector = SelectKBest(f_classif, k=2000)
X_train_selected = selector.fit_transform(train_X, train_y)
X_test_selected = selector.transform(test_X)
X_train_data = torch.tensor(X_train_selected, dtype=torch.float32)
X_test_data = torch.tensor(X_test_selected, dtype=torch.float32)
y_train_data = torch.tensor(train_y, dtype=torch.long)
y_test_data = torch.tensor(test_y, dtype=torch.long)

## Model Architecture and Training
Simple feedforward network: 2,000 → 128 → 6 with ReLU activation 
and dropout for regularization. Trained with cross-entropy loss 
and SGD-style manual updates.

In [ ]:
from torch import nn
import torch
import torch.nn.functional as F
lr = 0.005
epochs = 300
bs = 32
loss_func = F.cross_entropy

class genomic_id(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(2000, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.lin2 = nn.Linear(128, 6)
    
    def forward(self, x):
        x = self.lin1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.lin2(x)
        return x

model = genomic_id()
n = X_train_data.shape[0]

def fit():
    for epoch in range(epochs):
        model.train()
        for i in range((n - 1) // bs + 1):
            start_i = i * bs
            end_i = start_i +bs
            xb = X_train_data[start_i:end_i].float()
            yb = y_train_data[start_i:end_i]
            pred = model(xb)
            loss = loss_func(pred, yb)
            loss.backward()
            with torch.no_grad():
                for p in model.parameters():
                    p -= p.grad * lr
                model.zero_grad()
            print(f"epoch {epoch}, loss {loss.item():.4f}")
fit()
print(loss_func(model(xb), yb))

## Evaluation
Switching model to eval mode (disables dropout) and computing 
accuracy on the held-out test set.

In [ ]:
with torch.no_grad():
    model.eval()
    preds = model(X_test_data).argmax(dim=1)
    accuracy = (preds == y_test_data).float().mean()
    print(f"Test accuracy: {accuracy:.4f}")